# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## Task Type: **SIGNAL ANALYSIS**

Ranking Signal Analysis is primarily an observational signal-analysis task, not a supervised ranking model. The goal is to measure which safe content and search signals travel with visibility, clicks, engagement, or movement, then turn those measurements into a report for human review. A later model may be a secondary diagnostic, but it is not the task definition.

### Problem Statement

For a content editor or SEO analyst, determine which safe, observable content and search signals are associated with differences in visibility, clicks, engagement, and movement across content items.

### Why This Task Type?

- **Input:** One pseudonymized content item with content metadata and aggregated search/analytics signals.
- **Output:** A signal report containing grouped summaries, association sizes, and practical review recommendations.
- **Goal:** Identify signals worth investigating first while keeping the claims observational.
- **Not a ranking model:** The report ranks signals or review leads for attention; it does not claim to rank Google's results or predict Google's algorithm.

### Real-World Action This Supports

An editor or SEO analyst uses the report to choose which content groups or pages to inspect first, then decides whether to investigate content depth, search demand, position, engagement context, or recent movement.

In [1]:
task_definition = {
    "lane": "Ranking Signal Analysis",
    "task_type": "observational signal analysis",
    "unit": "one pseudonymized content item",
    "output": "signal report with grouped comparisons, effect sizes, and review recommendations",
    "primary_action": "help an editor choose what to investigate first",
}
for name, value in task_definition.items():
    print(f"{name}: {value}")
assert task_definition["task_type"] == "observational signal analysis"
assert task_definition["unit"] == "one pseudonymized content item"

lane: Ranking Signal Analysis
task_type: observational signal analysis
unit: one pseudonymized content item
output: signal report with grouped comparisons, effect sizes, and review recommendations
primary_action: help an editor choose what to investigate first


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or proxy

This lane has **no single predictive target**. Its primary outputs are measured associations with observed outcomes:

- visibility: `impressions_90d` and `avg_position`;
- demand response: `clicks_90d` and `ctr`;
- engagement: `sessions_90d`, `engagement_rate`, and `scroll_rate`;
- movement: `trend_direction` and `trend_pct`, treated as outcomes/labels and never as features.

These are observed aggregates, not causal labels. Any later classification or regression is a secondary analysis whose target and time alignment must be stated separately.

### What Are We Measuring?

The primary analysis will compare safe feature groups such as content type, intent, content age, word count, search demand, and prior-window activity against the observed outcome fields above. It will report medians or rates by group, rank correlations for numeric signals, and standardized effect sizes where appropriate.

### Why These Outcomes?

- They are present in the starter dataset and can be measured now.
- They connect directly to editor decisions about visibility and engagement review.
- They preserve the distinction between an observed outcome and a rule-defined product score.
- They allow practical recommendations without pretending that association proves causation.

`trend_direction` and `trend_pct` will be used only as movement outcomes or label-related audit fields. IDs and any product decision flags remain context or exclusions.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

The primary success criterion is **effect-size and grouped-summary quality**, not a made-up accuracy threshold. A useful result will identify stable, interpretable signal differences, report their magnitude, and show whether the pattern is consistent across client groups or sensitive to missingness. For any secondary predictive experiment, the metric must be declared with the target and evaluated using a client-grouped split.

### How Do We Measure Good Signal Analysis?

**Primary measures:**

- grouped medians or rates for interpretable categories;
- Spearman correlation for monotonic numeric associations;
- standardized mean difference or a comparable effect size for two-group comparisons;
- client-level consistency checks so one client does not dominate the finding.

**Decision standard:** a signal is worth recommending only when its direction and magnitude are clear enough to guide human review, its missingness is understood, and the caveat is stated. Weak correlations remain weak findings.

**Validation discipline:** use `client_id` for grouping and auditing only, never as a feature. Do not use label-derived fields, product scores, or future/overlapping windows as inputs. Results are observational and should be checked on held-out clients if a secondary model is trained.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

DATA_CANDIDATES = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
local_path = next((path for path in DATA_CANDIDATES if path.exists()), None)
if local_path is not None:
    df = pd.read_csv(local_path)
    data_source = str(local_path)
else:
    data_url = "https://raw.githubusercontent.com/somnathsutra/ML-01-ASSIGNMENT/main/data/raw/content_refresh_anonymized.csv"
    with urlopen(data_url) as response:
        df = pd.read_csv(BytesIO(response.read()))
    data_source = data_url

unit_columns = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "sessions_90d", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "trend_direction",
]

print("UNIT OF ANALYSIS: ONE ROW = ONE PSEUDONYMIZED CONTENT ITEM")
print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns")
print(f"Distinct content items: {df['content_id'].nunique():,}")
print(f"Distinct clients for grouped checks: {df['client_id'].nunique():,}")
print("\nExample rows, with identifiers omitted:")
print(df[unit_columns[2:]].head(3).to_string(index=False))
print("\nMissing values in displayed fields:")
print(df[unit_columns[2:]].isna().sum().to_string())

assert len(df) == 30_000
assert df["content_id"].nunique() == len(df)
assert df[["content_id", "client_id", "content_type", "trend_direction"]].notna().all().all()
print("\nUnit-of-analysis checks passed; descriptive and numeric missingness is retained for later handling.")

UNIT OF ANALYSIS: ONE ROW = ONE PSEUDONYMIZED CONTENT ITEM
Loaded 30,000 rows and 44 columns
Distinct content items: 30,000
Distinct clients for grouped checks: 32

Example rows, with identifiers omitted:
   content_type   main_intent  impressions_90d  clicks_90d  sessions_90d  ctr  avg_position  engagement_rate  scroll_rate trend_direction
keyword article transactional             3803          29            17 0.76          10.6             5.88         4.55            down
keyword article informational            15320           7             9 0.05          20.3             0.00        10.00            down
keyword article informational            12581          11            11 0.09          36.5             0.00        28.57            down

Missing values in displayed fields:
content_type          0
main_intent        2374
impressions_90d       0
clicks_90d            0
sessions_90d          0
ctr                   0
avg_position          0
engagement_rate       0
scroll_rate   

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why a signal report beats a fixed rule here

A fixed threshold can flag one condition, but this lane asks a broader question: which signals are associated with several observed outcomes, and how large or consistent are those differences? Search demand, content structure, position, engagement, and movement can vary together, while missingness follows content type. A single hand-written rule would hide those interactions and make it difficult to compare signal strength.

The analysis therefore earns its place by making the evidence inspectable: grouped summaries show the pattern, correlations and effect sizes quantify it, and client-level checks test whether the finding is driven by one group. A model may be used later as a diagnostic, but no model is allowed to turn an association into a causal claim.

In [5]:
# Verify the framing has measurable outcomes and a usable grouped-validation key.
required_outcomes = {
    "impressions_90d", "clicks_90d", "sessions_90d", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "trend_direction",
}
missing_outcomes = sorted(required_outcomes - set(df.columns))
client_counts = df.groupby("client_id").size()

print(f"Measured outcomes available: {len(required_outcomes) - len(missing_outcomes)}/{len(required_outcomes)}")
print(f"Missing outcomes: {missing_outcomes}")
print(f"Client groups: {client_counts.size}")
print(f"Smallest client group: {client_counts.min():,} rows")
print(f"Largest client group: {client_counts.max():,} rows")
print(f"Observed movement categories: {sorted(df['trend_direction'].astype('string').unique().tolist())}")

assert not missing_outcomes
assert client_counts.size == 32
assert client_counts.min() > 0
assert df["trend_direction"].astype("string").nunique() >= 2
print("ML-task framing checks passed.")

Measured outcomes available: 8/8
Missing outcomes: []
Client groups: 32
Smallest client group: 3 rows
Largest client group: 7,008 rows
Observed movement categories: ['down', 'flat', 'new', 'stable', 'up']
ML-task framing checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.

## 6. Self-check

| Requirement | Status | Details |
|---|---|---|
| **ML task type** | Complete | Observational signal analysis for the Ranking Signal Analysis lane |
| **Target/proxy** | Complete | No single predictive target; observed visibility, clicks, engagement, and movement outcomes |
| **Success metric** | Complete | Grouped summaries, correlations, effect sizes, and client-level consistency checks |
| **Unit of analysis** | Complete | One row = one pseudonymized content item |
| **Why this approach** | Complete | Multiple signals and patterned missingness need transparent comparisons, not one threshold |
| **Action connected** | Complete | Editors use the report to choose what to investigate first |
| **Data loaded** | Complete | Public-safe starter CSV, verified with executable checks |

The claims remain observational, directional, and decision-support only.